# Notebook 03 — Graph, Plain RW, Multi-Action Head (+ tuning)

PDF -> CDF -> temporal edges -> transition P -> **per-class plain RW** -> layered head -> mAP/F1.

Tuning changes baked in:
- **Data-driven sigma**: the PDF kernel bandwidth is set from the *median pairwise distance per window* (the toy `0.395` was only right for the 2-D slide; on 512-D features it collapses the kernel).
- **Knobs** exposed: `N`, `RW_STEPS`, `GAMMA`, plus a **diagnostics** cell and an **N-sweep** driver.
- Shape/range checks per section; metrics pooled **globally** over non-seed frames.

In [ ]:
# ===== CONFIG / KNOBS =====
import numpy as np
from pathlib import Path
np.set_printoptions(precision=3,suppress=True)
OUT_DIR=Path('./artifacts')

FEAT_NORM='center'    # per-window feature norm before the graph: 'none' | 'center' | 'standardize'
SIGMA_MODE='median'   # 'median' -> per-window median distance; or a float for a fixed bandwidth
GAMMA=0.90            # CDF cumulative-keep threshold
DECODER='fbeta'       # set decoder: 'fbeta' (existing per-class thresholds, DEFAULT) or 'rankgap' (experimental)
RANK_PERSIST=2        # rank-gap only: rank persistence radius in frames (0=off)
SPARSIFY='cdf'        # graph pruning rule: 'cdf' (cumulative-mass, GAMMA) or 'knn' (fixed top-k)
KNN_K=10              # neighbors kept per frame when SPARSIFY=='knn'
RW_STEPS=2            # plain-RW propagation steps
W_SMOOTH=5            # temporal smoothing window (7.3)
LAMBDA=0.3            # co-occurrence boost (7.4)
TOPK=0               # 0 = no forced labels (honest eval)
TOPK_FLOOR=0.5
MAX_K=3              # cap labels per frame (None to disable) -> kills 5-8 label over-firing
FBETA=0.5            # threshold fitting: <1 favors precision (fewer, cleaner labels); 1.0 = F1
ACT_PCTL=10          # background gate: frames below this pctl of seed max-score predict nothing (None=off)
ROW_CENTER=True      # subtract each frame's across-class mean -> removes 'activity' bias, keeps which-action signal
USE_CDF=True; USE_TEMPORAL=True   # ablation toggles (graph): CDF sparsification / temporal edges
FEAT_SUFFIX='' 
print('knobs:',dict(FEAT_NORM=FEAT_NORM,SIGMA_MODE=SIGMA_MODE,GAMMA=GAMMA,SPARSIFY=SPARSIFY,KNN_K=KNN_K,RW_STEPS=RW_STEPS,W_SMOOTH=W_SMOOTH,LAMBDA=LAMBDA,TOPK=TOPK,MAX_K=MAX_K,FBETA=FBETA,ACT_PCTL=ACT_PCTL,ROW_CENTER=ROW_CENTER))

## Stage 3 — PDF (data-driven sigma)

In [ ]:
def normalize_features(F):
    # Per-window: remove the common 'DC' scene component so similarity reflects action variation.
    if FEAT_NORM=='none': return F
    F=F-F.mean(0,keepdims=True)                       # mean-center
    if FEAT_NORM=='standardize': F=F/(F.std(0,keepdims=True)+1e-8)
    return F
def cosine_similarity(F):
    Fn=F/(np.linalg.norm(F,axis=1,keepdims=True)+1e-12); return Fn@Fn.T
def estimate_sigma(D):
    iu=np.triu_indices_from(D,k=1); d=D[iu]; s=np.median(d) if d.size else 1.0
    return float(s) if s>1e-6 else 1.0
def pdf_weights(F,sigma=None):
    S=cosine_similarity(F); D=1.0-S
    if sigma is None: sigma=estimate_sigma(D) if SIGMA_MODE=='median' else float(SIGMA_MODE)
    W=(1.0/(np.sqrt(2*np.pi)*sigma))*np.exp(-(D**2)/(2*sigma**2))
    return S,D,W,sigma

## Stage 4 — CDF + temporal edges + transition matrix

In [ ]:
def row_normalize(M): return M/(M.sum(1,keepdims=True)+1e-12)
def cdf_sparsify(W,gamma=GAMMA):
    P=row_normalize(W); keep=np.zeros_like(W,bool)
    for i in range(W.shape[0]):
        o=np.argsort(-P[i]); cs=np.cumsum(P[i][o]); k=np.searchsorted(cs,gamma)+1; keep[i,o[:k]]=True
    return W*keep
def knn_sparsify(W,k=None):
    # Fixed top-k pruning baseline: keep the k strongest neighbors of every frame.
    # Selection on the row-normalized affinities (same ordering as raw W); everything
    # downstream (temporal edges, column normalization) is identical to the CDF path,
    # so the comparison isolates the pruning rule alone.
    if k is None: k=KNN_K
    P=row_normalize(W); keep=np.zeros_like(W,bool)
    for i in range(W.shape[0]):
        keep[i,np.argsort(-P[i])[:min(k,W.shape[0])]]=True
    return W*keep
def sparsify(W,gamma=None):
    if gamma is None: gamma=GAMMA
    return knn_sparsify(W) if SPARSIFY=='knn' else cdf_sparsify(W,gamma)
def add_temporal_edges(A,wt=1.0):
    A=A.copy()
    for i in range(A.shape[0]-1): A[i,i+1]=max(A[i,i+1],wt); A[i+1,i]=max(A[i+1,i],wt)
    return A
def transition_matrix(A): return A/(A.sum(0,keepdims=True)+1e-12)

## Stage 6 — Plain RW (one independent walk per class)

In [ ]:
def random_walk(P,p0,steps=RW_STEPS):
    p=p0.astype(float).copy()
    for _ in range(steps): p=P@p
    return p
def stationary(P,steps=200): return random_walk(P,np.ones(P.shape[0])/P.shape[0],steps)
def propagate(P,Y,seed_mask,steps=RW_STEPS):
    n,C=Y.shape; sc=np.zeros((n,C)); si=np.where(seed_mask)[0]
    for cc in range(C):
        cls=[i for i in si if Y[i,cc]>0]
        if not cls: continue
        p0=np.zeros(n); p0[cls]=1.0/len(cls); sc[:,cc]=random_walk(P,p0,steps)
    return sc

## (Optional) Math-correctness check — reproduces the PDF slide exactly
Not dataset data: verifies the graph functions match the slide's S/D/W (with the slide's fixed sigma=0.395) and the plain-RW trace.

In [ ]:
F_toy=np.array([[1.,0.],[0.9,0.1],[0.,1.],[0.2,0.8]])
S,D,W,sg=pdf_weights(F_toy,sigma=0.395)
assert np.allclose(S,[[1,.994,0,.243],[.994,1,.110,.348],[0,.110,1,.970],[.243,.348,.970,1]],atol=2e-3)
assert np.allclose(W[0,0],1.010,atol=2e-3) and np.allclose(W[0,2],0.041,atol=2e-3)
P_pdf=np.array([[.295,.395,0,0],[.650,.169,.405,.072],[.013,.395,.191,.656],[.043,.042,.405,.272]])
p1=P_pdf@np.array([1.,0,0,0]); p2=P_pdf@p1
assert np.allclose(p1,[.295,.650,.013,.043],atol=2e-3) and np.allclose(p2,[.344,.310,.291,.057],atol=3e-3)
print('PDF S/D/W + RW trace reproduced (sigma=0.395)')

## Stage 7 — Layered head (global thresholds + co-occurrence)

In [ ]:
def h71_stationary(scores,P): return scores-stationary(P)[:,None]
def h72_calibrate(scores):
    mu=scores.mean(0,keepdims=True); sd=scores.std(0,keepdims=True)+1e-12; return (scores-mu)/sd
def h73_smooth(scores,w=W_SMOOTH):
    w=min(w,scores.shape[0])                 # clamp to window length (fixes N<W_SMOOTH)
    if w<=1: return scores
    k=np.ones(w)/w; out=np.empty_like(scores)
    for cc in range(scores.shape[1]): out[:,cc]=np.convolve(scores[:,cc],k,mode='same')
    return out
def cooccurrence(Ys):
    Co=Ys.T@Ys; np.fill_diagonal(Co,0.0); return Co/(Co.max()+1e-12)
def h74_cooccur(scores,Co,lam=LAMBDA): return scores+lam*(scores@Co.T)
def head_transform(raw,P,Co):
    s=h71_stationary(raw,P); s=h72_calibrate(s); s=h73_smooth(s); s=h74_cooccur(s,Co)
    if ROW_CENTER: s=s-s.mean(1,keepdims=True)   # per-frame centering: drop the common activity level
    return s
def h75_threshold(scores,taus,topk=None,floor=None,max_k=None,act_gate=None):
    # resolve knobs at CALL time (so editing the config cell takes effect without redefining funcs)
    if topk is None: topk=TOPK
    if floor is None: floor=TOPK_FLOOR
    if max_k is None: max_k=MAX_K
    n,C=scores.shape; pred=(scores>=taus[None,:]).astype(int)
    if topk>0:
        for i in range(n):
            if pred[i].sum()==0 and scores[i].max()>=floor: pred[i,np.argsort(-scores[i])[:topk]]=1
    if max_k:                                   # cap labels/frame -> keep top max_k by score among predicted
        for i in range(n):
            if pred[i].sum()>max_k:
                pidx=np.where(pred[i]==1)[0]; keep=pidx[np.argsort(-scores[i,pidx])[:max_k]]
                pred[i]=0; pred[i,keep]=1
    if act_gate is not None:                    # background gate: low-activity frames predict nothing
        pred[scores.max(1)<act_gate]=0
    return pred
def fit_thresholds(scores,Y,beta=None,grid=np.linspace(-2,3,51)):
    if beta is None: beta=FBETA          # resolve at CALL time (beta<1 favors precision)
    b2=beta*beta; C=scores.shape[1]; taus=np.full(C,np.inf)
    for cc in range(C):
        if Y[:,cc].sum()==0: continue
        best=-1
        for t in grid:
            pr=(scores[:,cc]>=t).astype(int)
            tp=(pr*Y[:,cc]).sum(); fp=(pr*(1-Y[:,cc])).sum(); fn=((1-pr)*Y[:,cc]).sum()
            fb=(1+b2)*tp/((1+b2)*tp + b2*fn + fp + 1e-12)   # F-beta
            if fb>best: best,taus[cc]=fb,t
    return taus

## Metrics — mAP + micro/macro F1 (global, nan-safe)

In [ ]:
def average_precision(yt,ys):
    o=np.argsort(-ys); y=yt[o]; tp=np.cumsum(y); fp=np.cumsum(1-y)
    prec=tp/(tp+fp+1e-12); rec=tp/(y.sum()+1e-12); ap=0.0; prev=0.0
    for p_,r_ in zip(prec,rec): ap+=p_*(r_-prev); prev=r_
    return ap
def evaluate(pred,scores,Y):
    present=np.where(Y.sum(0)>0)[0]
    if len(present)==0: return dict(mAP=float('nan'),micro_f1=float('nan'),macro_f1=float('nan'))
    mAP=np.mean([average_precision(Y[:,cc],scores[:,cc]) for cc in present])
    tp=(pred*Y).sum(); fp=(pred*(1-Y)).sum(); fn=((1-pred)*Y).sum(); micro=2*tp/(2*tp+fp+fn+1e-12)
    f1s=[]
    for cc in present:
        tpc=(pred[:,cc]*Y[:,cc]).sum(); fpc=(pred[:,cc]*(1-Y[:,cc])).sum(); fnc=((1-pred[:,cc])*Y[:,cc]).sum()
        f1s.append(2*tpc/(2*tpc+fpc+fnc+1e-12))
    return dict(mAP=float(mAP),micro_f1=float(micro),macro_f1=float(np.mean(f1s)),per_class_ap={int(cc):float(average_precision(Y[:,cc],scores[:,cc])) for cc in present})

def topk_accuracy(scores,Y,ks=(1,5)):
    # multi-label top-k: a frame is a hit if ANY true label is in its top-k scored classes.
    # scored over labeled frames only (background frames have no correct class).
    labeled=Y.sum(1)>0; s,y=scores[labeled],Y[labeled]
    if len(y)==0: return {f'top{k}':float('nan') for k in ks}
    order=np.argsort(-s,axis=1); out={}
    for k in ks:
        kk=min(k,s.shape[1]); topk=order[:,:kk]
        hit=np.array([y[i,topk[i]].any() for i in range(len(y))])
        out[f'top{kk}']=float(hit.mean())
    return out

def single_label_accuracy(scores,Y):
    # Single-label protocol: each frame's ONE prediction = top-scoring class.
    # Correct if that top class is among the frame's true labels. Scored on labeled frames only.
    labeled=Y.sum(1)>0; s,y=scores[labeled],Y[labeled]
    if len(y)==0: return dict(top1_acc=float('nan'),balanced_acc=float('nan'))
    top=np.argmax(s,axis=1)
    hit=np.array([y[i,top[i]]>0 for i in range(len(y))])
    top1=float(hit.mean())
    # balanced (per-class) single-label accuracy: mean over classes of recall@top1
    present=np.where(y.sum(0)>0)[0]; recs=[]
    for c in present:
        idx=np.where(y[:,c]>0)[0]
        recs.append(float(np.mean([y[i,top[i]]>0 and top[i]==c for i in idx])) if len(idx) else 0.0)
    return dict(top1_acc=top1, balanced_acc=float(np.mean(recs)))

## Loaders + runner (global pooling). Builds windows for any N from per-frame arrays.

In [ ]:
import pickle
def load_participant(P):
    feats=np.load(OUT_DIR/f'{P}_feats{FEAT_SUFFIX}.npz')['feats']
    d=np.load(OUT_DIR/f'{P}_data.npz',allow_pickle=True)
    return feats,d['targets'],d['seeds']
def make_windows(n,N,stride): return [list(range(s,s+N)) for s in range(0,n-N+1,stride)]
def run_participant(P,N=None,steps=RW_STEPS,gamma=GAMMA,verbose=True):
    feats,Y,seeds=load_participant(P); n,C=Y.shape
    wins=make_windows(n,N,N) if N else [list(w) for w in np.load(OUT_DIR/f'{P}_data.npz',allow_pickle=True)['windows']]
    Co=cooccurrence(Y[seeds]); Shead=np.zeros((n,C)); covered=np.zeros(n,bool); sigmas=[]
    for w in wins:
        idx=np.array(w,int)
        fw=normalize_features(feats[idx])             # per-window centering
        S,D,W,sg=pdf_weights(fw); sigmas.append(sg)
        A=sparsify(W,gamma) if USE_CDF else W
        A=add_temporal_edges(A) if USE_TEMPORAL else A
        P=transition_matrix(A)
        Shead[idx]=head_transform(propagate(P,Y[idx],seeds[idx],steps),P,Co); covered[idx]=True
    taus=fit_thresholds(Shead[seeds],Y[seeds]); ev=covered&(~seeds)
    pred=(h75_threshold(Shead[ev],taus) if DECODER=='fbeta' else decode_rankgap(Shead[ev])); m=evaluate(pred,Shead[ev],Y[ev])
    m.update(topk_accuracy(Shead[ev],Y[ev],ks=(1,5)))   # multi-label top-k (any true in top-k)
    m.update(single_label_accuracy(Shead[ev],Y[ev]))    # single-label top-1 (argmax) for ADL SOTA
    m['_frames']=np.where(ev)[0]; m['_pred']=pred; m['_scores']=Shead[ev]; m['_Y']=Y[ev]   # keep labels
    if verbose:
        print(f'{P} | N={N or "saved"} steps={steps} gamma={gamma} | sigma med={np.median(sigmas):.3f}')
        print('  ',{k:(round(v,4) if isinstance(v,float) else v) for k,v in m.items() if k!='per_class_ap'})
    return m

def run_pooled(participants,N=None,steps=RW_STEPS,gamma=GAMMA,verbose=True):
    # Graphs are built PER participant (temporal/similarity edges only make sense within one video),
    # but co-occurrence + thresholds are fit on POOLED seeds, and metrics pooled over all non-seed frames.
    # Requires a consistent class map across participants -> run NB01 with all of them in PARTICIPANTS.
    data={}; seedY=[]
    for p in participants:
        feats,Y,seeds=load_participant(p); data[p]=(feats,Y,seeds); seedY.append(Y[seeds])
    Cs={data[p][1].shape[1] for p in participants}
    assert len(Cs)==1, f'inconsistent class counts {Cs} - rebuild NB01 over all participants together'
    Co=cooccurrence(np.vstack(seedY))                      # GLOBAL co-occurrence across participants
    allS=[]; allY=[]; sScores=[]; sY=[]
    for p in participants:
        feats,Y,seeds=data[p]; n,C=Y.shape
        wins=make_windows(n,N,N) if N else [list(w) for w in np.load(OUT_DIR/f'{p}_data.npz',allow_pickle=True)['windows']]
        Shead=np.zeros((n,C)); covered=np.zeros(n,bool)
        for w in wins:
            idx=np.array(w,int); fw=normalize_features(feats[idx])
            S,D,W,sg=pdf_weights(fw)
            A=sparsify(W,gamma) if USE_CDF else W
            A=add_temporal_edges(A) if USE_TEMPORAL else A
            P=transition_matrix(A)
            Shead[idx]=head_transform(propagate(P,Y[idx],seeds[idx],steps),P,Co); covered[idx]=True
        ev=covered&(~seeds)
        allS.append(Shead[ev]); allY.append(Y[ev]); sScores.append(Shead[seeds]); sY.append(Y[seeds])
    seedScores=np.vstack(sScores); taus=fit_thresholds(seedScores,np.vstack(sY))  # GLOBAL thresholds on pooled seeds
    act=np.percentile(seedScores.max(1),ACT_PCTL) if ACT_PCTL is not None else None
    S=np.vstack(allS); Yv=np.vstack(allY)
    pred=(h75_threshold(S,taus,act_gate=act) if DECODER=='fbeta' else decode_rankgap(S,act_gate=act)); m=evaluate(pred,S,Yv); m.update(topk_accuracy(S,Yv,ks=(1,5)))
    if verbose:
        print(f'POOLED {participants} | N={N or "saved"} steps={steps} gamma={gamma} | eval frames={len(S)}')
        print('  ',{k:(round(v,4) if isinstance(v,float) else v) for k,v in m.items() if k!='per_class_ap'})
    return m

## Protocol 1 — per-participant, then mean (headline evaluation)

In [ ]:
import pickle
def run_all(participants, N=150, steps=10, gamma=0.9, verbose=True):
    """Protocol 1: run each participant at a FIXED config, then average metrics across participants
    (equal weight per participant). Per-class AP is averaged only over participants where the class
    appears. Requires a shared class map (build NB01 over all participants)."""
    cmap=pickle.load(open(OUT_DIR/'class_map.pkl','rb')); names=cmap['names']
    keys=['mAP','micro_f1','macro_f1','top1','top5','top1_acc','balanced_acc']  # top1_acc/balanced_acc = single-label
    rows={}; per_class={}
    for pp in participants:
        m=run_participant(pp,N=N,steps=steps,gamma=gamma,verbose=False)
        rows[pp]={k:m[k] for k in keys}
        for c,ap in m['per_class_ap'].items(): per_class.setdefault(int(c),[]).append(ap)
    hdr=f'{"participant":<14}'+''.join(f'{k:>10}' for k in keys)
    print(f'config: N={N} steps={steps} gamma={gamma} | {len(participants)} participants\n')
    print(hdr)
    for pp in participants: print(f'{pp:<14}'+''.join(f'{rows[pp][k]:>10.4f}' for k in keys))
    mean={k:float(np.mean([rows[pp][k] for pp in participants])) for k in keys}
    print('-'*len(hdr)); print(f'{"MEAN":<14}'+''.join(f'{mean[k]:>10.4f}' for k in keys))
    print('\nper-class AP (mean over participants where the class appears):')
    for c in sorted(per_class):
        aps=per_class[c]; print(f'  {names[c][:24]:<26} AP={np.mean(aps):.3f}  (n={len(aps)})')
    return dict(per_participant=rows, mean=mean, per_class={c:float(np.mean(v)) for c,v in per_class.items()})

# Headline protocol: fix the config, average across participants
res = run_all(['P_09','P_10', 'P_11','P_12', 'P_16','P_17'], N=150, steps=10, gamma=0.9)
print('\nHEADLINE mean mAP =', round(res['mean']['mAP'],4))

## See & save the (multi-)action labels

In [ ]:
# ---- Decode, view, and SAVE the (multi-)action labels ----

import csv
def predicted_label_sets(P, m=None, N=None):
    """Return per-frame predicted label sets (names) for the evaluated (non-seed) frames.
    Pass a metrics dict m from run_participant(...) to reuse it, else it is recomputed."""
    cmap=pickle.load(open(OUT_DIR/'class_map.pkl','rb')); names=cmap['names']
    if m is None: m=run_participant(P,N=N,verbose=False)
    frames,pred,sc,Yt=m['_frames'],m['_pred'],m['_scores'],m['_Y']
    rows=[]
    for r in range(len(frames)):
        pl=[int(c) for c in np.where(pred[r]==1)[0]]; tl=[int(c) for c in np.where(Yt[r]==1)[0]]
        rows.append(dict(frame=int(frames[r]),
                         pred_ids=pl, pred_names=[names[c] for c in pl], n_pred=len(pl),
                         true_ids=tl, true_names=[names[c] for c in tl], n_true=len(tl),
                         is_multi_pred=len(pl)>=2, is_multi_true=len(tl)>=2))
    return rows, names

def save_labels(P, m=None, N=None, only_multi=False):
    rows,names=predicted_label_sets(P,m=m,N=N)
    out=OUT_DIR/f'{P}_predicted_labels{"_multi" if only_multi else ""}.csv'
    sel=[r for r in rows if (r['is_multi_pred'] if only_multi else True)]
    with open(out,'w',newline='') as f:
        w=csv.writer(f); w.writerow(['frame','n_pred','pred_names','n_true','true_names','is_multi_pred','is_multi_true'])
        for r in sel:
            w.writerow([r['frame'],r['n_pred'],'|'.join(r['pred_names']),
                        r['n_true'],'|'.join(r['true_names']),int(r['is_multi_pred']),int(r['is_multi_true'])])
    n_multi=sum(r['is_multi_pred'] for r in rows)
    print(f'saved {len(sel)} rows -> {out}')
    print(f'  multi-action predicted frames: {n_multi}/{len(rows)} ({100*n_multi/max(len(rows),1):.1f}%)')
    return out

def show_multi_examples(P, m=None, N=None, k=15):
    rows,_=predicted_label_sets(P,m=m,N=N)
    multi=[r for r in rows if r['is_multi_pred']]
    print(f'{P}: {len(multi)} frames predicted with >=2 actions. First {min(k,len(multi))}:')
    for r in multi[:k]:
        tag='OK' if set(r['pred_ids'])&set(r['true_ids']) else '--'
        print(f"  frame {r['frame']:>6}: pred={r['pred_names']}  | true={r['true_names']}  [{tag}]")

# usage:
# m=run_participant('P_11')          # run once, reuse m below
# show_multi_examples('P_11', m=m)   # see multi-action frames in-notebook
# save_labels('P_11', m=m)           # save ALL predicted label sets
# save_labels('P_11', m=m, only_multi=True)   # save ONLY the multi-action frames

## Diagnostics — where is the signal being lost?
Seed coverage per window, sigma distribution, and per-class AP. Run after NB01+NB02 exist.

In [ ]:
def diagnostics(P,N=None):
    feats,Y,seeds=load_participant(P); n,C=Y.shape
    wins=make_windows(n,N,N) if N else [list(w) for w in np.load(OUT_DIR/f'{P}_data.npz',allow_pickle=True)['windows']]
    seedcounts=[int(seeds[np.array(w,int)].sum()) for w in wins]
    sig=[pdf_weights(normalize_features(feats[np.array(w,int)]))[3] for w in wins]
    print(f'{P}: {len(wins)} windows | zero-seed windows: {sum(s==0 for s in seedcounts)}'
          f' ({100*sum(s==0 for s in seedcounts)/len(wins):.1f}%)')
    print(f'  seeds/window: mean={np.mean(seedcounts):.2f} min={min(seedcounts)} max={max(seedcounts)}')
    print(f'  sigma/window: mean={np.mean(sig):.3f} min={np.min(sig):.3f} max={np.max(sig):.3f}')
    m=run_participant(P,N=N,verbose=False)
    print('  per-class AP:')
    for cc,ap in sorted(m['per_class_ap'].items(),key=lambda x:x[1]):
        print(f'    class {cc}: AP={ap:.3f}')
    try:
        import matplotlib.pyplot as plt
        fig,ax=plt.subplots(1,2,figsize=(11,3))
        ax[0].hist(seedcounts,bins=20); ax[0].set_title('seeds per window'); ax[0].set_xlabel('# seeds')
        ax[1].hist(sig,bins=20); ax[1].set_title('sigma per window'); ax[1].set_xlabel('sigma')
        plt.tight_layout(); plt.show()
    except Exception as e: print('plot skipped:',e)
    return m
# diagnostics('P_11')

## Run + N-sweep (the main in-pipeline tuning lever)

In [ ]:
# Single run on the saved windows:
# run_participant('P_11')

# Window-size sweep (also the window-size ablation):
# for N in [30,100,150,200]:
#     run_participant('P_11', N=N)

# RW-steps sweep at a chosen N:
# for t in [1,2,3,5,10]:
#     run_participant('P_11', N=150, steps=t)

# Pooled across participants (run NB01 with all of them so the class map is shared):
# run_pooled(['P_10','P_11'])

## Ablation study (ADL) — variants table + pruning sweep


In [ ]:
PARTICIPANTS=['P_09','P_10','P_11','P_12','P_16','P_17']
def ablation_study(units, N=100, steps=10, gamma=0.9):
    """ADL column of the variants table: FULL / -row-centering / -CDF / -temporal.
    Feature-level ablations (w/o DEFT, backbones, single-modality) need NB02 re-extraction,
    then set FEAT_SUFFIX and call run_all."""
    global ROW_CENTER,USE_CDF,USE_TEMPORAL
    save=(ROW_CENTER,USE_CDF,USE_TEMPORAL)
    configs={
        'FULL (all on)'    :(True ,True ,True ),
        '- row-centering'  :(False,True ,True ),
        '- CDF sparsify'   :(True ,False,True ),
        '- temporal edges' :(True ,True ,False),
    }
    keys=['mAP','macro_f1','micro_f1','top1']; res={}
    for name,(rc,cd,tm) in configs.items():
        ROW_CENTER,USE_CDF,USE_TEMPORAL=rc,cd,tm
        m=run_all(units,N=N,steps=steps,gamma=gamma,verbose=False)['mean']
        res[name]={k:m[k] for k in keys}
    ROW_CENTER,USE_CDF,USE_TEMPORAL=save
    full=res['FULL (all on)']
    print(f'ABLATION (ADL, N={N}, steps={steps}, {len(units)} participants) — mean over participants\n')
    print(f'{"config":<20}'+''.join(f'{k:>10}' for k in keys)+f'{"dmAP":>9}')
    for name in configs:
        r=res[name]; d=r['mAP']-full['mAP']
        print(f'{name:<20}'+''.join(f'{r[k]:>10.4f}' for k in keys)+f'{d:>+9.4f}')
    return res

def pruning_sweep(units, N=100, steps=10, ks=(5,10,15,20), gammas=(0.80,0.85,0.90,0.95)):
    """ADL half of the KNN-vs-CDF pruning table."""
    global SPARSIFY,KNN_K
    save=(SPARSIFY,KNN_K); rows=[]
    for k in ks:
        SPARSIFY='knn'; KNN_K=k
        m=run_all(units,N=N,steps=steps,verbose=False)['mean']; rows.append(('KNN',k,m))
    SPARSIFY='cdf'
    for g in gammas:
        m=run_all(units,N=N,steps=steps,gamma=g,verbose=False)['mean']; rows.append(('CDF',g,m))
    SPARSIFY,KNN_K=save
    print(f'\nPRUNING SWEEP (ADL, N={N}, steps={steps}, {len(units)} participants)\n')
    print(f'{"method":<8}{"value":>7}{"mAP":>9}{"top1":>9}{"macroF1":>10}{"microF1":>10}')
    for meth,val,m in rows:
        print(f'{meth:<8}{val:>7}{m["mAP"]:>9.4f}{m["top1"]:>9.4f}{m["macro_f1"]:>10.4f}{m["microF1" if False else "micro_f1"]:>10.4f}')
    return rows

# ablation_study(PARTICIPANTS, N=100, steps=10)
# pruning_sweep(PARTICIPANTS, N=100, steps=10)


### Qualitative timeline figure (GT vs SMART, per class) — vector output for the paper


In [ ]:
def timeline_figure(P_name, m=None, N=100, classes=None, save_path=None, formats=('svg','eps')):
    """Qualitative GT-vs-prediction timeline for one participant.
    save_path: path WITHOUT extension (e.g. OUT_DIR/'timeline_P_11'); saves one file per format.
    Vector formats (svg/eps/pdf) stay sharp at any LaTeX scale. EPS has no transparency -> no alpha used."""
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    GT_COLOR   = '#3A3A3A'   # dark gray reference
    PRED_COLOR = '#0072B2'   # Okabe-Ito blue: colorblind-safe, grayscale-distinct
    BAND_COLOR = '#EDEDED'   # alternating row band
    cmap_names = pickle.load(open(OUT_DIR/'class_map.pkl','rb'))['names']
    if m is None: m = run_participant(P_name, N=N, verbose=False)
    frames, pred, Yt = m['_frames'], m['_pred'], m['_Y']
    n = frames.max()+1
    def tracks(mat):
        T = np.zeros((n, mat.shape[1])); T[frames] = mat; return T
    GT, PR = tracks(Yt), tracks(pred)
    if classes is None: classes = [c for c in range(GT.shape[1]) if GT[:,c].sum() > 0]
    fig, ax = plt.subplots(figsize=(11, 0.52*len(classes)+1))
    def spans(track):
        d = np.diff(np.concatenate([[0], track, [0]]))
        return list(zip(np.where(d==1)[0], np.where(d==-1)[0]))
    for r, c in enumerate(classes):
        y = len(classes)-1-r
        if r % 2 == 0:
            ax.axhspan(y-0.5, y+0.5, color=BAND_COLOR, zorder=0)
        for s, e in spans(GT[:,c]): ax.barh(y+0.19, e-s, left=s, height=0.34, color=GT_COLOR,   zorder=2)
        for s, e in spans(PR[:,c]): ax.barh(y-0.19, e-s, left=s, height=0.34, color=PRED_COLOR, zorder=2)
    ax.set_yticks(range(len(classes)))
    ax.set_yticklabels([cmap_names[c][:26] for c in reversed(classes)], fontsize=9)
    ax.set_xlabel('frame', fontsize=10); ax.set_xlim(0, n); ax.set_ylim(-0.5, len(classes)-0.5)
    ax.tick_params(axis='x', labelsize=9)
    for sp in ['top','right','left']: ax.spines[sp].set_visible(False)
    ax.legend(handles=[mpatches.Patch(color=GT_COLOR,   label='Ground truth'),
                       mpatches.Patch(color=PRED_COLOR, label='SMART')],
              loc='upper right', fontsize=9, framealpha=1.0)
    ax.set_title(f'{P_name}: per-frame multi-action timeline', fontsize=11)
    plt.tight_layout()
    if save_path:
        save_path = str(save_path)
        for ext in formats:
            out = f'{save_path}.{ext}'
            plt.savefig(out, format=ext, bbox_inches='tight',
                        **({} if ext in ('svg','eps','pdf') else {'dpi':300}))
            print('saved ->', out)
    plt.show()

# ADL showcase candidate: P_11 (the row-centering breakthrough participant).
# If it has too many GT classes to render, pass classes=[...] with a hand-picked subset.
# m = run_participant('P_11', N=100, verbose=False)
# timeline_figure('P_11', m=m, save_path=OUT_DIR/'timeline_P_11', formats=('svg','eps'))


## Rank-gap decoder + ranking metrics (EXPERIMENTAL, additive — DECODER='fbeta' keeps everything unchanged)


In [ ]:
def decode_rankgap(scores, max_k=None, act_gate=None, persist=None):
    """Rank-gap set decoding: per frame, sort classes by row-centered score and cut the
    label set at the largest consecutive score gap within the first max_k ranks.
    Optional rank persistence: a candidate survives only if it appears in the top
    region for a majority of frames in a (2*persist+1) window. Uses NO thresholds."""
    if max_k is None: max_k=(MAX_K or 3)
    if persist is None: persist=RANK_PERSIST
    n,C=scores.shape; pred=np.zeros((n,C),int)
    order=np.argsort(-scores,axis=1); kmax=min(max_k,C-1)
    for i in range(n):
        s=scores[i,order[i]]
        k=int(np.argmax(s[:kmax]-s[1:kmax+1]))+1     # cut at largest gap
        pred[i,order[i,:k]]=1
    if persist and persist>0:
        top=np.zeros((n,C))
        for i in range(n): top[i,order[i,:kmax]]=1
        cs=np.vstack([np.zeros((1,C)),np.cumsum(top,axis=0)])
        keep=np.zeros((n,C))
        for i in range(n):
            a=max(0,i-persist); b=min(n,i+persist+1)
            keep[i]=(cs[b]-cs[a])/(b-a)              # fraction of window frames with class in top region
        pred=(pred*(keep>0.5)).astype(int)
    if act_gate is not None: pred[scores.max(1)<act_gate]=0
    return pred

def ranking_metrics(scores,Y):
    """Label-ranking quality of the scores themselves (decoder-independent).
    LRAP: for each positive label, precision of positives within the ranks above it (higher=better).
    Coverage: how deep in the ranking one must go to cover all true labels (lower=better)."""
    n,C=scores.shape; lraps=[]; covs=[]
    for i in range(n):
        pos=np.where(Y[i]==1)[0]
        if len(pos)==0: continue
        rank=np.empty(C); rank[np.argsort(-scores[i])]=np.arange(1,C+1)
        r=rank[pos]
        lraps.append(np.mean([ (r<=rj).sum()/rj for rj in r ]))
        covs.append(r.max())
    return dict(LRAP=float(np.mean(lraps)), coverage=float(np.mean(covs)))

def compare_decoders(units, N=100, steps=10, gamma=0.9):
    """SAFE comparison driver: runs run_all under DECODER='fbeta' then 'rankgap' and
    prints both. Restores DECODER afterwards. Run the regression check first!"""
    global DECODER
    save=DECODER; out={}
    for dec in ['fbeta','rankgap']:
        DECODER=dec
        m=run_all(units,N=N,steps=steps,gamma=gamma,verbose=False)['mean']
        out[dec]={k:m[k] for k in ['mAP','macro_f1','micro_f1','top1']}
    DECODER=save
    print(f'\nDECODER COMPARISON (N={N}, steps={steps}, {len(units)} units)\n')
    print(f'{"decoder":<10}{"mAP":>9}{"macroF1":>10}{"microF1":>10}{"top1":>9}')
    for dec,m in out.items():
        print(f'{dec:<10}{m["mAP"]:>9.4f}{m["macro_f1"]:>10.4f}{m["micro_f1"]:>10.4f}{m["top1"]:>9.4f}')
    print('note: mAP/top1 are decoder-independent (score-based) and must be IDENTICAL across rows;')
    print('      only macro/micro-F1 (set-based) can differ. If mAP differs, something is wrong — stop.')
    return out

# --- SAFETY / REGRESSION CHECK: run FIRST, with DECODER='fbeta' (default) ---
# m=run_all(PARTICIPANTS,N=100,steps=10,verbose=False)['mean']; print(m['mAP'])
# This MUST reproduce your locked FULL number exactly. Only then try:
# compare_decoders(PARTICIPANTS, N=100, steps=10)
